# AyeAI Physical AI Platform - Simulation GPU Qualification Lab

**Copyright © 1993-2026 Abhishek Choudhary. All rights reserved.**  
**AyeAI® · AyeAM® · AyeCNSe®** are registered trademarks of Abhishek Choudhary.  
**Package:** `ayeai-physical-ai-simulation-dropin` · **Version:** 3.1.0 · **Platform baseline:** Iteration 47  
**License:** GNU GPL v3.0 for package-authored materials. NVIDIA Warp, MuJoCo/MJX, JAX and other dependencies retain their own licenses. See `THIRD-PARTY-NOTICES.md` in the package.

This notebook is designed for a **browser-only workflow**. Run it in Google Colab; no local IDE, Docker, or local GPU installation is required. Select **Runtime → Change runtime type → GPU** before running the GPU cells. Colab GPU availability depends on account quota and current capacity.

## What this notebook qualifies

1. Runtime discovery and explicit CPU/GPU device reporting.
2. NVIDIA Warp GPU kernel execution with a bounded particle-gravity/floor-contact microbenchmark and CPU reference comparison.
3. MuJoCo MJX articulated-body execution using JAX devices, with GPU used when available and CPU fallback reported distinctly.
4. Numerical sanity checks, deterministic inputs, runtime metadata, and machine-readable evidence export.

**Important boundary:** these are separate Python research/qualification paths. They are not silently substituted for the package's deterministic 2D disc solver, and the notebook does not claim that a GPU backend is already registered in the Iteration 47 synchronous browser simulator. A production bridge requires an explicit async worker/host boundary, schema mapping, identity/provenance, cancellation, timeout, and HOLD semantics as described in `GROK-HANDOFF.md`.

### Open in Colab

- [Open Google Colab](https://colab.research.google.com/) and upload this `.ipynb` from the Files panel, or use **File → Upload notebook**.
- After the repository is published, Grok should replace the template below with the real repository path: `https://colab.research.google.com/github/<OWNER>/<REPOSITORY>/blob/main/drop-in/simulation/notebooks/simulation-gpu-qualification.ipynb`.
- Do not publish a fabricated GitHub Colab URL before the notebook is committed to that exact path.


## 0. Runtime and evidence policy

This notebook does not request credentials, open inbound ports, expose a runtime server, or upload project data. The output evidence file contains only runtime versions, device names, test outcomes, and aggregate benchmark values. Review it before sharing.

The notebook treats missing GPU devices or unsupported optional packages as `SKIP`, not `PASS`. A CPU fallback is labelled CPU. A GPU result is only a successful kernel execution and numerical sanity check, not evidence of physical fidelity or real-time performance.


In [ ]:
import os, sys, json, time, platform, importlib.util
from pathlib import Path
import numpy as np

EVIDENCE = {
    "package": "ayeai-physical-ai-simulation-dropin",
    "package_version": "3.1.0",
    "iteration": 47,
    "copyright": "Copyright © 1993-2026 Abhishek Choudhary. All rights reserved.",
    "trademarks": ["AyeAI®", "AyeAM®", "AyeCNSe®"],
    "license": "GPL-3.0 for package-authored materials; third-party licenses remain separate",
    "python": sys.version,
    "platform": platform.platform(),
    "numpy": np.__version__,
    "tests": {},
    "limitations": [
        "GPU microbenchmarks are not physical validation",
        "Warp particle kernel is not equivalent to the native rigid-disc contact solver",
        "MJX execution is a separate articulated-body backend path",
        "No production registration or live platform bridge is performed by this notebook"
    ]
}
print(json.dumps({k:EVIDENCE[k] for k in ["package","package_version","iteration","python","platform","numpy"]}, indent=2))


## 1. Install optional GPU-capable engines

Warp is a Python GPU simulation framework with CPU and CUDA execution. Current NVIDIA wheels may require a recent CUDA driver and GPU architecture; if installation succeeds but CUDA is unavailable, Warp can still run on CPU. MJX is MuJoCo's JAX-based implementation and supports NVIDIA/AMD GPUs and Google Cloud TPUs through XLA. Installation is optional and independently reported.


In [ ]:
%pip -q install "warp-lang==1.18.0" "mujoco==3.17.0" "mujoco-mjx==3.17.0"


In [ ]:
# Runtime discovery - never infer GPU availability from the Colab menu alone.
import importlib
runtime = {"cuda_nvidia_smi": False, "warp": "NOT_INSTALLED", "jax_devices": [], "mujoco_mjx": "NOT_INSTALLED"}
try:
    import subprocess
    q = subprocess.run(["nvidia-smi", "--query-gpu=name,driver_version,memory.total", "--format=csv,noheader"], capture_output=True, text=True, timeout=10)
    runtime["cuda_nvidia_smi"] = q.returncode == 0
    runtime["nvidia_smi"] = q.stdout.strip() if q.returncode == 0 else q.stderr.strip()
except Exception as exc:
    runtime["nvidia_smi"] = f"UNAVAILABLE: {type(exc).__name__}: {exc}"
try:
    import warp as wp
    wp.init()
    runtime["warp"] = {"version": getattr(wp, "__version__", "unknown"), "devices": [str(d) for d in wp.get_devices()]}
except Exception as exc:
    runtime["warp"] = f"UNAVAILABLE: {type(exc).__name__}: {exc}"
try:
    import jax
    runtime["jax_devices"] = [f"{d.platform}:{d.device_kind}" for d in jax.devices()]
except Exception as exc:
    runtime["jax_devices"] = [f"UNAVAILABLE: {type(exc).__name__}: {exc}"]
try:
    import mujoco, mujoco.mjx
    runtime["mujoco_mjx"] = {"mujoco": mujoco.__version__, "mjx": "importable"}
except Exception as exc:
    runtime["mujoco_mjx"] = f"UNAVAILABLE: {type(exc).__name__}: {exc}"
EVIDENCE["runtime"] = runtime
print(json.dumps(runtime, indent=2))


## 2. NVIDIA Warp - bounded GPU kernel qualification

The microbenchmark integrates independent particles under constant gravity with a floor clamp. It is intentionally small and analytically checkable. It tests kernel compilation, device execution, finite-state behavior, and CPU-reference agreement. It does **not** implement pairwise rigid-body contact, articulated joints, or the native package's accumulated-impulse contact solver.


In [ ]:
warp_result = {"status": "SKIP", "reason": "Warp unavailable"}
try:
    import warp as wp
    wp.init()
    @wp.kernel
    def integrate_floor(pos: wp.array(dtype=wp.vec3), vel: wp.array(dtype=wp.vec3), dt: float, gravity: wp.vec3):
        i = wp.tid()
        v = vel[i] + gravity * dt
        p = pos[i] + v * dt
        if p[1] < 0.0:
            p = wp.vec3(p[0], 0.0, p[2])
            if v[1] < 0.0:
                v = wp.vec3(v[0], 0.0, v[2])
        pos[i] = p
        vel[i] = v

    n = 4096
    dt = 1.0 / 240.0
    steps = 240
    rng = np.random.default_rng(47001)
    p0 = np.zeros((n, 3), dtype=np.float32)
    p0[:, 0] = rng.uniform(-1.0, 1.0, n)
    p0[:, 1] = rng.uniform(0.5, 5.0, n)
    v0 = np.zeros((n, 3), dtype=np.float32)
    v0[:, 0] = rng.uniform(-0.1, 0.1, n)
    wp_pos = wp.array(p0, dtype=wp.vec3, device="cuda" if wp.is_cuda_available() else "cpu")
    wp_vel = wp.array(v0, dtype=wp.vec3, device=wp_pos.device)
    device_name = str(wp_pos.device)
    start = time.perf_counter()
    for _ in range(steps):
        wp.launch(integrate_floor, dim=n, inputs=[wp_pos, wp_vel, dt, wp.vec3(0.0, -9.81, 0.0)], device=wp_pos.device)
    wp.synchronize_device(wp_pos.device)
    elapsed = time.perf_counter() - start
    final_pos, final_vel = wp_pos.numpy(), wp_vel.numpy()
    assert np.isfinite(final_pos).all() and np.isfinite(final_vel).all()
    assert np.all(final_pos[:, 1] >= -1e-6)
    warp_result = {"status": "PASS", "device": device_name, "device_class": "GPU" if "cuda" in device_name.lower() else "CPU", "particles": n, "steps": steps, "elapsed_seconds": elapsed, "particle_steps_per_second": n*steps/max(elapsed,1e-12), "finite_state": True, "floor_nonpenetration": True}
except Exception as exc:
    warp_result = {"status": "SKIP", "reason": f"{type(exc).__name__}: {exc}"}
EVIDENCE["warp_microbenchmark"] = warp_result
print(json.dumps(warp_result, indent=2))


## 3. MuJoCo MJX - articulated rigid-body qualification

This section builds a one-link pendulum model, transfers its model/data to the JAX device, and advances a bounded number of simulation steps. It reports the actual JAX device. On a CPU-only runtime this remains a CPU test and must not be represented as GPU execution.


In [ ]:
mjx_result = {"status": "SKIP", "reason": "MuJoCo MJX unavailable"}
try:
    import mujoco
    from mujoco import mjx
    import jax
    import jax.numpy as jnp
    xml = r'''<mujoco model="pap_mjx_pendulum">
      <option timestep="0.002" gravity="0 0 -9.81" integrator="implicitfast"/>
      <worldbody>
        <body name="pendulum" pos="0 0 0">
          <joint name="hinge" type="hinge" axis="0 1 0" damping="0.02"/>
          <geom type="capsule" fromto="0 0 0 0 0 -1" size="0.04" mass="1"/>
        </body>
      </worldbody>
    </mujoco>'''
    model = mujoco.MjModel.from_xml_string(xml)
    data = mujoco.MjData(model)
    data.qpos[0] = 0.4
    mjx_model = mjx.put_model(model)
    mjx_data = mjx.put_data(model, data)
    step_fn = jax.jit(mjx.step)
    start = time.perf_counter()
    for _ in range(500):
        mjx_data = step_fn(mjx_model, mjx_data)
    jax.block_until_ready(mjx_data.qpos)
    elapsed = time.perf_counter() - start
    qpos = np.asarray(mjx_data.qpos)
    qvel = np.asarray(mjx_data.qvel)
    assert np.isfinite(qpos).all() and np.isfinite(qvel).all()
    mjx_result = {"status":"PASS", "device":str(jax.devices()[0]), "device_class":jax.devices()[0].platform.upper(), "steps":500, "elapsed_seconds":elapsed, "final_qpos":qpos.tolist(), "final_qvel":qvel.tolist(), "finite_state":True}
except Exception as exc:
    mjx_result = {"status":"SKIP", "reason":f"{type(exc).__name__}: {exc}"}
EVIDENCE["mujoco_mjx"] = mjx_result
print(json.dumps(mjx_result, indent=2))


## 4. Acceptance summary and evidence export

A `PASS` below means only that the specific software check completed. `SKIP` is not a pass. The notebook does not convert these checks into physical verification or platform backend qualification.


In [ ]:
checks = {"warp_microbenchmark": EVIDENCE.get("warp_microbenchmark",{}).get("status","SKIP"), "mujoco_mjx": EVIDENCE.get("mujoco_mjx",{}).get("status","SKIP")}
EVIDENCE["summary"] = {"checks":checks, "pass_count":sum(v=="PASS" for v in checks.values()), "skip_count":sum(v=="SKIP" for v in checks.values()), "physical_verification":False, "production_qualification":False}
output = Path('/content/simulation-gpu-qualification-evidence.json')
output.write_text(json.dumps(EVIDENCE, indent=2, ensure_ascii=False), encoding='utf-8')
print(json.dumps(EVIDENCE["summary"], indent=2))
print(f"Evidence written: {output}")


## 5. Grok integration checklist

1. Keep this notebook at `drop-in/simulation/notebooks/simulation-gpu-qualification.ipynb` and preserve its metadata, copyright, trademarks, license notice, and evidence limitations.
2. Add the real GitHub Colab URL only after committing the notebook to the repository path. Until then, link to the Colab home page and instruct users to upload the notebook.
3. Treat Warp and MJX as external Python execution backends. Do not claim that either is wired into the browser simulator merely because the notebook runs.
4. For production use, implement a separately reviewed host adapter implementing the package's versioned JSON-RPC contract. Use a worker/job boundary, request IDs, authentication, origin allowlisting, quotas, cancellation, timeouts, schema validation, provenance, deterministic replay inputs, and explicit HOLD on any mismatch.
5. Keep remote engine execution asynchronous. Iteration 47's `Simulator.step()` is synchronous and does not await promises. Do not block the browser main thread or fake a synchronous GPU backend.
6. Add live-engine numerical comparison tests against analytical solutions and independent reference cases. Pin exact engine, CUDA, driver, Python, JAX and Warp versions. Record hardware and compiler metadata.
7. Run package tests, all Iteration 47 regression tests, no-mutation acceptance, mutation acceptance, and browser acceptance after stitching. Do not alter the kernel or acceptance script in this package change.

### Sources

- NVIDIA Warp: https://nvidia.github.io/warp/stable/index.html
- MuJoCo MJX: https://mujoco.readthedocs.io/en/latest/mjx.html
- Project Chrono GPU build support: https://api.chrono.projectchrono.org/tutorial_install_chrono.html
